<a href="https://colab.research.google.com/github/justverena/Circle_test_task/blob/main/notebooks/02_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes wandb

# Fine-Tuning Mistral-7B with QLoRA


Applying **QLoRA** using **Hugging Face Transformers** and **PEFT** to adapt the model with **Colab T4 GPU**.

## Steps
1. Loading the dataset (JSONL)
2. Tokenizer preparation
3. Formatting instruction-response pairs
4. Dataset tokenization
5. Loading base model
6. LoRA configuration
7. Model training and logging
8. Saving trained adapter

## Imports and W&B login

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import wandb
wandb.login()

## Dataset jsonl loading

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
dataset = load_dataset("json", data_files="dataset.jsonl")
dataset

## Tokenizer preparation

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

## Formatting instruction-response pairs

In [ ]:
def format_example(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

dataset = dataset["train"].map(format_example)

## Dataset tokenization

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

## Loading base model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

## LoRA configuration

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

## Training Arquments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    report_to="wandb",
    run_name="mistral-lora-finetune"
)

## Model training and logging (with data collector and trainer)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)
trainer.train()

## Saving trained adapter

In [ ]:
model.save_pretrained("mistral-lora-adapter")
tokenizer.save_pretrained("mistral-lora-adapter")